**ROLL NO : 24BAD087**  
**NAME : POOVIKA M**

**EXP NO : 9**

**TEXT CLASSIFICATION AND TEXT GENERATION USING TRANSFORMERS**

PART A - Install libraries and load dataset

In [1]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece

import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load IMDb dataset
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)
print("Sample review:", dataset['train'][0]['text'])
print("Sample label:", dataset['train'][0]['label'])

# Use smaller subsets for faster training
train_data = dataset['train'].shuffle(seed=42).select(range(2000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Tokenize dataset
train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

train_data = train_data.remove_columns(['text'])
test_data = test_data.remove_columns(['text'])

train_data.set_format('torch')
test_data.set_format('torch')

print("Tokenization completed!")

Device: cuda


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (9 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization completed!


PART B - Fine-tune BERT for text classification

In [2]:
# Load pretrained BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)

def compute_metrics(pred):
    labels = pred.label_ids
    predictions = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary',
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

# Train model
trainer.train()

# Evaluate model
results = trainer.evaluate()

print("Evaluation Results:")
print(results)

# Save model and tokenizer
trainer.save_model('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.439850,0.842000,0.771987,0.963415,0.857143
2,0.366155,0.442833,0.858000,0.837838,0.882114,0.859406


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.366155,0.442833,2,0.858000,0.837838,0.882114,0.859406


Evaluation Results:
{'eval_loss': 0.4428332448005676, 'eval_accuracy': 0.858, 'eval_precision': 0.8378378378378378, 'eval_recall': 0.8821138211382114, 'eval_f1': 0.8594059405940594}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_sentiment_model/tokenizer_config.json',
 './bert_sentiment_model/tokenizer.json')

PART C - Test BERT using new text inputs

In [3]:
# Load trained model
model = AutoModelForSequenceClassification.from_pretrained(
    './bert_sentiment_model'
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    './bert_sentiment_model'
)

texts = [
    "This movie is absolutely wonderful and amazing!",
    "This movie is terrible and very boring."
]

for text in texts:
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    label = "Positive" if prediction == 1 else "Negative"

    print("Input:", text)
    print("Predicted Sentiment:", label)
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Input: This movie is absolutely wonderful and amazing!
Predicted Sentiment: Positive

Input: This movie is terrible and very boring.
Predicted Sentiment: Negative



PART D - Text generation using T5 Transformer

In [4]:
# Load pretrained T5 model
t5_tokenizer = AutoTokenizer.from_pretrained(
    "google-t5/t5-small"
)

t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google-t5/t5-small"
).to(device)

# Sample input text
texts = [
    "summarize: Artificial intelligence is transforming many industries. "
    "It helps businesses automate tasks, analyze data, and improve "
    "decision-making. Machine learning and deep learning are important "
    "technologies used in artificial intelligence.",

    "summarize: Online education allows students to learn from anywhere. "
    "It provides flexibility, digital resources, and access to courses. "
    "However, students need internet access and self-discipline."
]

for text in texts:

    inputs = t5_tokenizer(
        text,
        return_tensors='pt',
        max_length=512,
        truncation=True
    ).to(device)

    outputs = t5_model.generate(
        **inputs,
        max_length=60,
        num_beams=4,
        early_stopping=True
    )

    generated_text = t5_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Input Text:")
    print(text)

    print("\nGenerated Summary:")
    print(generated_text)

    print("-" * 50)

print("Text generation completed!")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Input Text:
summarize: Artificial intelligence is transforming many industries. It helps businesses automate tasks, analyze data, and improve decision-making. Machine learning and deep learning are important technologies used in artificial intelligence.

Generated Summary:
artificial intelligence is transforming many industries. it helps businesses automate tasks, analyze data and improve decision-making.
--------------------------------------------------
Input Text:
summarize: Online education allows students to learn from anywhere. It provides flexibility, digital resources, and access to courses. However, students need internet access and self-discipline.

Generated Summary:
online education provides flexibility, digital resources, and access to courses. however, students need internet access and self-discipline.
--------------------------------------------------
Text generation completed!
